### Initial Setup

In [1]:
# CONFIG
COCO_JSON_PATH = "/content/instances_COCO.json"
IMAGES_DIR = "/content"
OUTPUT_DIR = "/content/output_white_bg"

In [2]:
!pip install opencv-python numpy pycocotools
from IPython.display import clear_output
clear_output()

In [3]:
import json
import os
import cv2
import numpy as np
import pycocotools.mask as maskUtils

In [4]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

### Helper Functions

In [5]:
def load_coco(json_path):
    with open(json_path, "r") as f:
        return json.load(f)

In [6]:
coco = load_coco(COCO_JSON_PATH)


images_dict = {img["id"]: img for img in coco["images"]}
categories_dict = {cat["id"]: cat["name"] for cat in coco["categories"]}
# Create output subdirectories per category
for cat_id, cat_name in categories_dict.items():
    os.makedirs(os.path.join(OUTPUT_DIR, cat_name), exist_ok=True)

for idx, ann in enumerate(coco["annotations"]):
    img_info = images_dict[ann["image_id"]]
    img_path = os.path.join(IMAGES_DIR, img_info["file_name"])

    img = cv2.imread(img_path)
    if img is None:
        print(f"⚠️ Could not read image: {img_path}")
        continue

    h, w = img.shape[:2]
    mask = np.zeros((h, w), dtype=np.uint8)

    segmentation = ann.get("segmentation")
    shape_type = ann.get("attributes", {}).get("type", "polygon")

    # -------------------------------------------------
    # CASE 1: segmentation is a LIST
    # -------------------------------------------------
    if isinstance(segmentation, list) and len(segmentation) > 0:

        for seg in segmentation:

            # RLE inside list (rare but valid)
            if isinstance(seg, dict):
                rle = seg
                if isinstance(rle["counts"], list):
                    rle = maskUtils.frPyObjects(rle, h, w)
                decoded = maskUtils.decode(rle)
                mask = np.maximum(mask, decoded.astype(np.uint8) * 255)

            elif isinstance(seg, list):
                flat = np.array(seg, dtype=np.float32)
                pts = flat.reshape(-1, 2)

                # ✅ Ellipse: [cx, cy, rx, ry, angle] → 5 flat values
                if len(flat) == 5 or shape_type == "ellipse":
                    try:
                        cx, cy, rx, ry, angle = flat[:5]
                        cv2.ellipse(
                            mask,
                            (int(cx), int(cy)),
                            (int(abs(rx)), int(abs(ry))),
                            float(angle),
                            0, 360,
                            255, -1
                        )
                    except Exception as e:
                        print(f"⚠️ Ellipse draw failed for ann {ann['id']}: {e}")

                # ✅ 2-point rectangle: [x1, y1, x2, y2] → 4 flat values or 2 pts
                elif len(pts) == 2 or (len(flat) == 4 and shape_type in ("rectangle", "points")):
                    x1, y1 = int(pts[0][0]), int(pts[0][1])
                    x2, y2 = int(pts[1][0]), int(pts[1][1])
                    # Normalize in case coords are flipped
                    x1, x2 = min(x1, x2), max(x1, x2)
                    y1, y2 = min(y1, y2), max(y1, y2)
                    cv2.rectangle(mask, (x1, y1), (x2, y2), 255, -1)

                # ✅ Standard polygon: 3+ points
                elif len(pts) >= 3:
                    cv2.fillPoly(mask, [pts.astype(np.int32)], 255)

                else:
                    print(f"⚠️ Skipping unrecognised seg shape (ann {ann['id']}): {len(flat)} values")

    # -------------------------------------------------
    # CASE 2: segmentation is a DICT (pure RLE)
    # -------------------------------------------------
    elif isinstance(segmentation, dict):
        rle = segmentation
        if isinstance(rle["counts"], list):
            rle = maskUtils.frPyObjects(rle, h, w)
        decoded = maskUtils.decode(rle)
        mask = decoded.astype(np.uint8) * 255

    # -------------------------------------------------
    # CASE 3: fallback to bbox
    # -------------------------------------------------
    else:
        print(f"ℹ️ No segmentation for ann {ann['id']}, falling back to bbox")
        x, y, bw, bh = map(int, ann["bbox"])
        cv2.rectangle(mask, (x, y), (x + bw, y + bh), 255, -1)

    # Skip if mask is empty
    if mask.max() == 0:
        print(f"⚠️ Empty mask for ann {ann['id']} — skipping")
        continue

    # Apply mask: keep original pixels inside mask, white outside
    white = np.ones_like(img) * 255
    result = np.where(mask[:, :, None] == 255, img, white)

    # Crop to bounding box of mask
    coords = cv2.findNonZero(mask)
    if coords is not None:
        x, y, bw, bh = cv2.boundingRect(coords)
        result = result[y:y + bh, x:x + bw]

    # Save to category subdirectory
    cat_name = categories_dict.get(ann["category_id"], "unknown")
    out_name = f"{cat_name}__ann{ann['id']}.png"
    out_path = os.path.join(OUTPUT_DIR, cat_name, out_name)
    cv2.imwrite(out_path, result)

print("✅ Extraction complete!")

ℹ️ No segmentation for ann 4, falling back to bbox
ℹ️ No segmentation for ann 9, falling back to bbox
ℹ️ No segmentation for ann 19, falling back to bbox
ℹ️ No segmentation for ann 21, falling back to bbox
ℹ️ No segmentation for ann 24, falling back to bbox
ℹ️ No segmentation for ann 25, falling back to bbox
ℹ️ No segmentation for ann 26, falling back to bbox
ℹ️ No segmentation for ann 27, falling back to bbox
ℹ️ No segmentation for ann 28, falling back to bbox
ℹ️ No segmentation for ann 111, falling back to bbox
ℹ️ No segmentation for ann 112, falling back to bbox
ℹ️ No segmentation for ann 113, falling back to bbox
ℹ️ No segmentation for ann 117, falling back to bbox
ℹ️ No segmentation for ann 124, falling back to bbox
ℹ️ No segmentation for ann 142, falling back to bbox
ℹ️ No segmentation for ann 143, falling back to bbox
ℹ️ No segmentation for ann 144, falling back to bbox
ℹ️ No segmentation for ann 146, falling back to bbox
ℹ️ No segmentation for ann 147, falling back to bbox
ℹ️ N

In [7]:
# from IPython.display import display
# import glob

# for img_path in glob.glob("/content/output_white_bg/*/*.png")[:5]:
#     display(Image.open(img_path))

In [8]:
# import shutil
# shutil.rmtree(OUTPUT_DIR)
# os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
!zip -r output_white_bg.zip output_white_bg/
from google.colab import files
files.download(f"{OUTPUT_DIR}.zip")